# CriterionLens one-time final test evaluation

Run this kernel only after the project is frozen and the code dataset has been synced from the exact `v1.0-frozen` tag. The four planned base/adapter variants are evaluated once each with greedy no-cache decoding and 64 new tokens. Do not retry any model listed in `results/TEST_ACCESS_LOG.md`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from packaging.version import Version

devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print({'devices': devices, 'torch': torch.__version__, 'architectures': torch.cuda.get_arch_list()})
if len(devices) < 2 or any('T4' not in name for name in devices):
    raise RuntimeError(f'This final evaluation requires two Tesla T4 GPUs; found {devices}')

candidates = sorted({
    path.parent
    for path in Path('/kaggle/input').rglob('requirements.txt')
    if (path.parent / 'pyproject.toml').is_file()
})
if len(candidates) != 1:
    raise RuntimeError(
        'Expected exactly one code dataset with requirements.txt and pyproject.toml; '
        f'found {candidates}'
    )
DATASET_DIR = candidates[0]
print(f'Found code dataset at: {DATASET_DIR}')
CODE_DIR = Path('/kaggle/working/code')
if CODE_DIR.exists():
    raise RuntimeError(f'Unexpected pre-existing work directory: {CODE_DIR}')
shutil.copytree(DATASET_DIR, CODE_DIR, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))

freeze_manifest = json.loads((CODE_DIR / 'freeze_manifest.json').read_text(encoding='utf-8'))
if freeze_manifest.get('tag') != 'v1.0-frozen':
    raise RuntimeError('Code dataset is not from the v1.0-frozen tag')
commit = freeze_manifest.get('commit')
valid_commit = isinstance(commit, str) and len(commit) == 40
valid_commit = valid_commit and all(c in '0123456789abcdef' for c in commit)
if not valid_commit:
    raise RuntimeError('Freeze manifest does not contain a full Git commit SHA')
print(f'Frozen source verified: {freeze_manifest["tag"]} at {commit}')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE_DIR / 'requirements.txt')],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'],
    cwd=CODE_DIR,
    check=True,
)

try:
    torchao_version = version('torchao')
except PackageNotFoundError:
    torchao_version = None
if torchao_version and Version(torchao_version) < Version('0.16.1'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)

OUT_DIR = Path('/kaggle/working/out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
os_env = os.environ.copy()
os_env['PYTHONPATH'] = str(CODE_DIR / 'src') + os.pathsep + os_env.get('PYTHONPATH', '')
print(f'Code directory: {CODE_DIR}; output directory: {OUT_DIR}')

In [ ]:
evaluation_runs = [
    ('base_1p5b', 'Qwen/Qwen2.5-1.5B-Instruct', None),
    (
        'ft_1p5b', 'Qwen/Qwen2.5-1.5B-Instruct',
        'Ashher123/criterion-lens-qwen2.5-1.5b-lora',
    ),
    ('base_0p5b', 'Qwen/Qwen2.5-0.5B-Instruct', None),
    (
        'ft_0p5b', 'Qwen/Qwen2.5-0.5B-Instruct',
        'Ashher123/criterion-lens-qwen2.5-0.5b-lora',
    ),
]

failures = []
for name, model_id, adapter_id in evaluation_runs:
    command = [
        sys.executable, 'scripts/evaluate.py',
        '--model', model_id,
        '--split', 'test', '--final',
        '--decoding', 'no_cache', '--max-new-tokens', '64',
        '--out', str(CODE_DIR / 'results' / 'eval' / f'test_{name}'),
    ]
    if adapter_id:
        command.extend(['--adapter', adapter_id])
    print(f'Running the one-time final evaluation: {name}', flush=True)
    try:
        subprocess.run(command, cwd=CODE_DIR, env=os_env, check=True)
    except subprocess.CalledProcessError as error:
        failures.append((name, error.returncode))
        print(f'Final evaluation failed for {name} with exit code {error.returncode}')

In [ ]:
import zipfile

archive_path = OUT_DIR / 'final_test_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    result_roots = [
        CODE_DIR / 'results' / 'eval',
        CODE_DIR / 'results' / 'TEST_ACCESS_LOG.md',
    ]
    for root in result_roots:
        if root.is_file():
            archive.write(root, root.relative_to(CODE_DIR))
        elif root.is_dir():
            for path in root.glob('test_*/*'):
                if path.is_file():
                    archive.write(path, path.relative_to(CODE_DIR))
print(f'Packaged one-time test results and access log at {archive_path}')
if failures:
    raise RuntimeError(f'One-time final evaluation failures: {failures}')